In [1]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
import nflreadpy as nfl

In [2]:
MIN_GAMES = 20
START_YEAR = 2024
END_YEAR = 2025
DEF_POSITIONS = {'DB', 'LB', 'DL'}
OF_POSITIONS = {'OL', 'WR', 'RB', 'TE'}
BIO_COLS = ['player_id', 'first_name', 'last_name', 'position', 'total_teams', 'current_team', 'games', 'seasons']
QB_COLS = ['completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'passing_epa',
           'carries', 'rushing_yards', 'rushing_tds', 'rushing_epa']
OF_COLS = ['carries', 'rushing_yards', 'rushing_tds', 'rushing_epa', 'receptions', 'targets',
           'receiving_yards', 'receiving_tds', 'receiving_epa']
DEF_COLS = ['def_tackles_solo', 'def_tackles_with_assist', 'def_tackle_assists', 'def_sacks', 'def_sack_yards',
            'def_qb_hits', 'def_interceptions']
ST_COLS = ['fg_made', 'fg_att', 'fg_long', 'fg_made_distance', 
           'pt_att', 'pt_inside_20', 'pt_downed', 'pt_touchback', 'pt_fair_caught', 'pt_yards']

raw = nfl.load_player_stats(seasons = list(range(START_YEAR, END_YEAR + 1)), summary_level = 'reg').to_pandas()
list(raw.columns)

['player_id',
 'player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'season',
 'season_type',
 'recent_team',
 'games',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'passing_interceptions',
 'sacks_suffered',
 'sack_yards_lost',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_cpoe',
 'passing_2pt_conversions',
 'pacr',
 'passing_10',
 'passing_16',
 'passing_20',
 'passing_40',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'rushing_10',
 'rushing_12',
 'rushing_20',
 'rushing_40',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_conversions',
 'receiv

In [3]:
raw.position_group.value_counts()

position_group
DB      821
DL      625
LB      611
OL      528
WR      477
RB      313
TE      264
SPEC    217
QB      159
Name: count, dtype: int64

Not all players have two word names (ex. Jordan van den Berg). I will make first name the first word and last name the rest of the words. 

In [4]:
def get_player_stats(data, position, cols):
    data = data.copy()

    positions = {position} if isinstance(position, str) else position

    stat_aggs = {col: (col, 'max' if col == 'fg_long' else 'sum') for col in cols}

    player_stats = (
        data
        .sort_values(by = 'season', ascending = True)
        .loc[data['position_group'].isin(positions)]
        .groupby('player_id', as_index = False)
        .agg(
            name = ('player_display_name', 'last'),
            position = ('position_group', 'last'),
            total_teams = ('recent_team', 'nunique'),
            current_team = ('recent_team', 'last'),
            games = ('games', 'sum'),
            seasons = ('games', 'count'),
            **stat_aggs
        )
    )

    player_stats['first_name'] = [name.split()[0] for name in player_stats['name']]
    player_stats['last_name'] = [" ".join(name.split()[1:]) for name in player_stats['name']]

    return player_stats[BIO_COLS + cols].loc[player_stats.games >= MIN_GAMES].reset_index(drop = True)


In [5]:
qb_stats = get_player_stats(raw, 'QB', QB_COLS)
of_stats = get_player_stats(raw, OF_POSITIONS, OF_COLS)
def_stats = get_player_stats(raw, DEF_POSITIONS, DEF_COLS)
st_stats = get_player_stats(raw, 'SPEC', ST_COLS)

In [6]:
qb_stats.head()

,player_id,first_name,last_name,position,total_teams,current_team,games,seasons,completions,attempts,passing_yards,passing_tds,passing_interceptions,passing_epa,carries,rushing_yards,rushing_tds,rushing_epa
0,00-0023459,Aaron,Rodgers,QB,2,PIT,33,2,695,1082,7219,52,18,39.619893,43,168,1,1.329357
1,00-0026158,Joe,Flacco,QB,2,CIN,20,2,413,664,4240,27,17,-38.819739,30,61,1,-1.578302
2,00-0026498,Matthew,Stafford,QB,1,LA,33,2,728,1114,8469,66,16,183.306725,59,42,0,-34.847156
3,00-0029604,Kirk,Cousins,QB,1,ATL,24,2,469,722,5229,28,21,20.203590,37,7,1,-19.264515
4,00-0030565,Geno,Smith,QB,2,LV,32,2,709,1026,7345,40,32,-51.168310,94,381,2,-0.432239


In [7]:
of_stats.head()

,player_id,first_name,last_name,position,total_teams,current_team,games,seasons,carries,rushing_yards,rushing_tds,rushing_epa,receptions,targets,receiving_yards,receiving_tds,receiving_epa
0,00-0029892,Kyle,Juszczyk,RB,1,SF,30,2,7,23,1,-10.787526,43,59,413,4,7.337932
1,00-0030035,Adam,Thielen,WR,2,PIT,24,2,0,0,0,0.000000,67,97,801,5,45.608586
2,00-0030061,Zach,Ertz,TE,1,WAS,30,2,0,0,0,0.000000,116,163,1158,11,42.692837
3,00-0030279,Keenan,Allen,WR,2,LAC,32,2,0,0,0,0.000000,151,243,1521,11,56.191363
4,00-0030506,Travis,Kelce,TE,1,KC,33,2,2,2,0,-1.967418,173,241,1674,8,61.039505


In [8]:
def_stats.head()

,player_id,first_name,last_name,position,total_teams,current_team,games,seasons,def_tackles_solo,def_tackles_with_assist,def_tackle_assists,def_sacks,def_sack_yards,def_qb_hits,def_interceptions
0,00-0026190,Calais,Campbell,DL,2,ARI,34,2,49,9,37,11.5,90.0,28,0
1,00-0027940,Von,Miller,LB,2,WAS,25,2,27,2,14,15.0,114.0,23,0
2,00-0027962,Cameron,Jordan,LB,1,NO,32,2,45,5,31,14.5,90.5,23,1
3,00-0027969,Cameron,Heyward,DL,1,PIT,34,2,65,8,76,11.5,78.0,29,0
4,00-0029255,Bobby,Wagner,LB,1,WAS,34,2,141,13,140,6.5,35.0,17,2


In [9]:
st_stats.head()

,player_id,first_name,last_name,position,total_teams,current_team,games,seasons,fg_made,fg_att,fg_long,fg_made_distance,pt_att,pt_inside_20,pt_downed,pt_touchback,pt_fair_caught,pt_yards
0,00-0025565,Nick,Folk,SPEC,2,NYJ,30,2,49,51,58.0,1981,0,0,0,0,0,0
1,00-0027114,Thomas,Morstead,SPEC,2,SF,31,2,0,0,NaN,0,107,41,10,8,31,4891
2,00-0028872,Johnny,Hekker,SPEC,2,TEN,34,2,0,0,NaN,0,151,46,26,10,41,6991
3,00-0029692,Bryan,Anger,SPEC,1,DAL,34,2,0,0,NaN,0,103,38,6,3,33,4999
4,00-0029822,Brandon,McManus,SPEC,1,GB,25,2,44,51,56.0,1547,0,0,0,0,0,0


In [10]:
player_list = (
    pd.concat([qb_stats, of_stats, def_stats, st_stats], ignore_index = True)[BIO_COLS]
    .drop_duplicates(subset = 'player_id')
    .reset_index(drop = True)
)

player_list.head()

,player_id,first_name,last_name,position,total_teams,current_team,games,seasons
0,00-0023459,Aaron,Rodgers,QB,2,PIT,33,2
1,00-0026158,Joe,Flacco,QB,2,CIN,20,2
2,00-0026498,Matthew,Stafford,QB,1,LA,33,2
3,00-0029604,Kirk,Cousins,QB,1,ATL,24,2
4,00-0030565,Geno,Smith,QB,2,LV,32,2


In [11]:
player_list.to_csv('data/player_list.csv', index = False)
stats = {'qb': qb_stats, 'of': of_stats, 'def': def_stats, 'st': st_stats}
for pos, df in stats.items():
    df.to_csv(f'data/{pos}_stats.csv', index = False)

## Player Evaluation